In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# Two reactions at once, twice over: a common ion and a coupled hydrolysis (Illustrations 13.3-2 and 13.3-3)

$$\mathrm{AgCl(s)}\rightleftharpoons\mathrm{Ag^+}+\mathrm{Cl^-}
\qquad
\mathrm{TlCl(s)}\rightleftharpoons\mathrm{Tl^+}+\mathrm{Cl^-}
\qquad\qquad
\mathrm{ATP}+\mathrm{H_2O}\rightleftharpoons\mathrm{ADP}+\mathrm{P_i}$$

Section 13.3 is about what happens when reactions cannot be solved one at a time. Its two
remaining illustrations are the two reasons they cannot.

| | why the reactions couple | what it costs |
|---|---|---|
| **13.3-2** | **a shared species.** Both salts release Cl$^-$, and both raise the ionic strength that sets $\gamma_{\pm}$ for each -- so they are coupled twice over, once through a mass balance and once through an activity coefficient | the solubility of AgCl falls by **three and a half orders of magnitude** |
| **13.3-3** | **a reaction too favorable to measure.** ATP hydrolysis has an equilibrium constant no experiment can pin down, so it is measured as the *sum* of two reactions that can be, and $K_a$ is their product (Eq. 13.3-10) | nothing, if the arithmetic is right -- and here it is not |

**Both illustrations carry an arithmetic defect, and both are found the same way**: by
recomputing what the illustration says it computed and substituting back. Neither changes
what the section teaches about coupling; one of them does change what its own closing
sentence claims.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from scipy.optimize import fsolve

from thermo.charts import use_book_style
use_book_style()

R = 8.314

# Eq. 9.10-18 -- the extended form Sec. 13.3 names for these ionic strengths. NOT the
# limiting law: I reaches 0.144 here, six times past where Sec. 13.2 saw the limiting law
# fail, so the 1 + sqrt(I) denominator and the linear salting term are both needed.
A_DH, B_SALT = 1.178, 0.1

def ln_gamma(I, z_product=1.0):
    """ln gamma_pm = -A |z+ z-| sqrt(I)/(1 + sqrt(I)) + B |z+ z-| I, Eq. 9.10-18."""
    r = np.sqrt(I)
    return z_product * (-A_DH * r / (1 + r) + B_SALT * I)

def gamma_pm(I, z_product=1.0):
    return np.exp(ln_gamma(I, z_product))

print(f"Eq. 9.10-18 with A = {A_DH}, B = {B_SALT}")
for I in (1e-5, 1e-3, 0.144):
    print(f"  I = {I:8.5f}   gamma_pm = {gamma_pm(I):.5f}")

Eq. 9.10-18 with A = 1.178, B = 0.1
  I =  0.00001   gamma_pm = 0.99629
  I =  0.00100   gamma_pm = 0.96463
  I =  0.14400   gamma_pm = 0.73370


## Illustration 13.3-2 -- the common ion effect

Silver chloride dissolves to $1.273\times10^{-5}$ kmol/m$^3$ in pure water; thallium
chloride to $0.144$. Both give chloride. What is the simultaneous solubility?

The two ideal-solution solubility products come from the two pure-salt measurements, each
with its own ionic strength -- and for a 1:1 salt dissolving in pure water,
$I=C_{\rm salt}$.

**The silver chloride number is the same $1.607\times10^{-10}$ that Illustration 13.2-3
extrapolates out of the Popoff and Neumann data.** Two different routes -- one from a single
solubility with an activity-coefficient correction, one from five solubilities extrapolated
to zero ionic strength -- and they agree to four figures. That is established before any
of the coupling starts.

In [3]:
S_AGCL, S_TLCL = 1.273e-5, 0.144        # kmol/m3 in pure water at 25 C

K_AGCL = S_AGCL ** 2 * gamma_pm(S_AGCL) ** 2
K_TLCL = S_TLCL ** 2 * gamma_pm(S_TLCL) ** 2
print(f"K0_AgCl = {K_AGCL:.4e} (kmol/m3)^2     printed 1.607e-10")
print(f"K0_TlCl = {K_TLCL:.4e} (kmol/m3)^2     printed 1.116e-02")
assert abs(K_AGCL - 1.607e-10)/1.607e-10 < 1e-3
assert abs(K_TLCL - 1.116e-2)/1.116e-2 < 1e-3

print(f"\nIllustration 13.2-3 got Ks0 = 1.6070e-10 for AgCl by extrapolating five")
print(f"solubilities to zero ionic strength. This route uses one solubility and a")
print(f"gamma correction, and gets {K_AGCL:.4e}. Two independent routes, four figures.")

K0_AgCl = 1.6070e-10 (kmol/m3)^2     printed 1.607e-10
K0_TlCl = 1.1163e-02 (kmol/m3)^2     printed 1.116e-02

Illustration 13.2-3 got Ks0 = 1.6070e-10 for AgCl by extrapolating five
solubilities to zero ionic strength. This route uses one solubility and a
gamma correction, and gets 1.6070e-10. Two independent routes, four figures.


### The simultaneous solution

Two equilibrium relations, one chloride balance, and an ionic strength that depends on the
answer:

$$K^{\circ}_{\rm AgCl}=C_{\rm Ag^+}C_{\rm Cl^-}\gamma_{\pm}^2
\qquad
K^{\circ}_{\rm TlCl}=C_{\rm Tl^+}C_{\rm Cl^-}\gamma_{\pm}^2
\qquad
C_{\rm Cl^-}=C_{\rm Ag^+}+C_{\rm Tl^+}
\qquad
I=C_{\rm Cl^-}$$

The illustration solves it by inspection -- assume the thallium chloride is unaffected,
because the silver chloride is so much less soluble -- and then checks the assumption. The
notebook solves the full system instead, and then checks that the illustration's shortcut
was justified. Those are different claims and the second is the interesting one.

In [4]:
def residuals(log_C):
    C_Ag, C_Tl = np.exp(log_C)
    C_Cl = C_Ag + C_Tl
    I = C_Cl                                   # 1/2 (C_Ag + C_Tl + C_Cl) with all |z| = 1
    g2 = gamma_pm(I) ** 2
    # Solved in logs: the two concentrations differ by eight orders of magnitude, and a
    # linear residual would be satisfied by any C_Ag at all.
    return [np.log(C_Ag * C_Cl * g2 / K_AGCL), np.log(C_Tl * C_Cl * g2 / K_TLCL)]

C_Ag, C_Tl = np.exp(fsolve(residuals, np.log([2e-9, 0.144]), full_output=False))
C_Cl = C_Ag + C_Tl
print(f"full simultaneous solution")
print(f"  C_Ag+ = {C_Ag:.4e} kmol/m3       printed 2.119e-09")
print(f"  C_Tl+ = {C_Tl:.6f} kmol/m3")
print(f"  C_Cl- = {C_Cl:.6f} kmol/m3        I = {C_Cl:.6f}")
print(f"  residuals: {np.max(np.abs(residuals(np.log([C_Ag, C_Tl])))):.2e}")

print(f"\nwas the shortcut justified?")
print(f"  TlCl solubility moved from {S_TLCL:.6f} to {C_Tl:.6f}"
      f"   -- {100*abs(C_Tl-S_TLCL)/S_TLCL:.1e} percent")
print(f"  AgCl solubility fell from {S_AGCL:.4e} to {C_Ag:.4e}"
      f"   -- a factor of {S_AGCL/C_Ag:.0f}, {np.log10(S_AGCL/C_Ag):.1f} orders of magnitude")

full simultaneous solution
  C_Ag+ = 2.0731e-09 kmol/m3       printed 2.119e-09
  C_Tl+ = 0.144000 kmol/m3
  C_Cl- = 0.144000 kmol/m3        I = 0.144000
  residuals: 1.89e-15

was the shortcut justified?
  TlCl solubility moved from 0.144000 to 0.144000   -- 6.4e-07 percent
  AgCl solubility fell from 1.2730e-05 to 2.0731e-09   -- a factor of 6141, 3.8 orders of magnitude


### B48 -- a transposed square root, and it moves three printed numbers

The full solve gives $C_{\rm Ag^+}=2.073\times10^{-9}$ where the illustration prints
$2.119\times10^{-9}$ -- 2.2 % apart, which is too large for rounding in a calculation
carried to four figures and too small to be a method difference.

**It is one transposed digit.** $\sqrt{0.144}=0.3795$, and the illustration prints
**0.3975** -- twice, once in the text and once inside the exponential of the NaNO$_3$
comment. The cell below settles which value each printed number was computed from, and the
answer is clean: $K^{\circ}_{\rm TlCl}$ was computed **before** the slip and the three
numbers after it were computed **with** it.

In [5]:
rt_true, rt_printed = np.sqrt(S_TLCL), 0.3975
print(f"sqrt(0.144) = {rt_true:.6f}      the illustration prints 0.3975, twice")

def with_root(rt):
    """Every downstream number, computed from a given value of sqrt(I) at I = 0.144."""
    g = np.exp(-A_DH * rt / (1 + rt) + B_SALT * S_TLCL)
    return dict(gamma=g,
                K_TlCl=S_TLCL ** 2 * g * g,
                C_Ag=K_AGCL / (S_TLCL * g * g),
                C_NaNO3=np.sqrt(K_AGCL * np.exp(2 * A_DH * rt / (1 + rt)
                                                - 2 * B_SALT * S_TLCL)))

PRINTED = dict(K_TlCl=1.116e-2, C_Ag=2.119e-9, C_NaNO3=1.75e-5)
rows = []
for lbl, rt in (("printed 0.3975", rt_printed), ("correct 0.3795", rt_true)):
    v = with_root(rt)
    rows.append((lbl, v["gamma"], v["K_TlCl"], v["C_Ag"], v["C_NaNO3"],
                 100 * (v["C_NaNO3"] / S_AGCL - 1)))
print(f"\n{'':>16}{'gamma':>9}{'K0_TlCl':>12}{'C_Ag+':>12}{'C_NaNO3':>12}{'increase':>10}")
for r in rows:
    print(f"{r[0]:>16}{r[1]:9.5f}{r[2]:12.4e}{r[3]:12.4e}{r[4]:12.4e}{r[5]:9.1f} %")
print(f"{'PRINTED':>16}{'':>9}{PRINTED['K_TlCl']:12.4e}{PRINTED['C_Ag']:12.4e}"
      f"{PRINTED['C_NaNO3']:12.4e}{37.0:9.1f} %")

bad, good = with_root(rt_printed), with_root(rt_true)
print("\nwhich root was each printed number computed from?")
for k in ("K_TlCl", "C_Ag", "C_NaNO3"):
    d_bad = abs(bad[k] - PRINTED[k]) / PRINTED[k]
    d_good = abs(good[k] - PRINTED[k]) / PRINTED[k]
    who = "0.3975 (the slip)" if d_bad < d_good else "0.3795 (correct)"
    print(f"  {k:9s} -> {who:20s}  gaps: {100*d_bad:.2f} % vs {100*d_good:.2f} %")

# The pattern is the finding: K0_TlCl was computed correctly and everything after it was
# not, which places the transposition after that line and before the next.
assert abs(good["K_TlCl"] - PRINTED["K_TlCl"]) / PRINTED["K_TlCl"] < 1e-3
assert abs(bad["C_Ag"] - PRINTED["C_Ag"]) / PRINTED["C_Ag"] < 1e-3
assert abs(bad["C_NaNO3"] - PRINTED["C_NaNO3"]) / PRINTED["C_NaNO3"] < 2e-3

sqrt(0.144) = 0.379473      the illustration prints 0.3975, twice

                    gamma     K0_TlCl       C_Ag+     C_NaNO3  increase
  printed 0.3975  0.72567  1.0919e-02  2.1193e-09  1.7469e-05     37.2 %
  correct 0.3795  0.73370  1.1163e-02  2.0731e-09  1.7278e-05     35.7 %
         PRINTED           1.1160e-02  2.1190e-09  1.7500e-05     37.0 %

which root was each printed number computed from?
  K_TlCl    -> 0.3795 (correct)      gaps: 2.16 % vs 0.02 %
  C_Ag      -> 0.3975 (the slip)     gaps: 0.01 % vs 2.17 %
  C_NaNO3   -> 0.3975 (the slip)     gaps: 0.18 % vs 1.27 %


### The comment: a salt with *no* common ion

If the 0.144 kmol/m$^3$ of thallium chloride is replaced by the same concentration of
sodium nitrate, there is no shared ion and the only coupling left is through the ionic
strength -- which pushes the other way. The solubility of silver chloride **rises**.

In [6]:
C_na = with_root(rt_true)["C_NaNO3"]
print(f"0.144 kmol/m3 NaNO3, no common ion:")
print(f"  C_AgCl = {C_na:.4e} kmol/m3     printed 1.75e-05, which carries the same slip")
print(f"  corrected, the increase is {100*(C_na/S_AGCL-1):.1f} percent, not 37")
print(f"\n0.144 kmol/m3 TlCl, common ion:")
print(f"  C_AgCl = {C_Ag:.4e} kmol/m3     a fall of {100*(1-C_Ag/S_AGCL):.4f} percent")
print(f"\nSame ionic strength, opposite sign, and a factor of"
      f" {C_na/C_Ag:.0f} between the two answers. The common ion is worth four orders")
print(f"of magnitude; the ionic strength is worth 36 percent.")

0.144 kmol/m3 NaNO3, no common ion:
  C_AgCl = 1.7278e-05 kmol/m3     printed 1.75e-05, which carries the same slip
  corrected, the increase is 35.7 percent, not 37

0.144 kmol/m3 TlCl, common ion:
  C_AgCl = 2.0731e-09 kmol/m3     a fall of 99.9837 percent

Same ionic strength, opposite sign, and a factor of 8334 between the two answers. The common ion is worth four orders
of magnitude; the ionic strength is worth 36 percent.


## Illustration 13.3-3 -- ATP hydrolysis, and the scatter that is not there

ATP hydrolysis is too favorable to measure directly. Rosing and Slater [*Biochim. Biophys.
Acta* **267**, 275 (1972)] measured two reactions that can be measured and took the product
of their constants -- Eq. 13.3-10 -- which is the section's point about intermediate
reactions made into an experiment.

They report $\Delta_{\rm rxn}G^{\circ}$ at 25 $^{\circ}$C and at 37 $^{\circ}$C, both
extrapolated to zero ionic strength, at five values of pH. The illustration asks for the
heat of reaction, which two Gibbs energies at two temperatures give through
Gibbs-Helmholtz:

$$\Delta_{\rm rxn}H^{\circ}=
\frac{\Delta_{\rm rxn}G^{\circ}(T_2)/T_2-\Delta_{\rm rxn}G^{\circ}(T_1)/T_1}
     {1/T_2-1/T_1}$$

**Three of the five printed heats reproduce exactly and two do not** -- and the two
that do not are the entire basis of the illustration's closing sentence.

In [7]:
T1, T2 = 298.15, 310.15                # 25 C and 37 C
pH   = np.array([6.0, 6.5, 7.0, 7.5, 8.0])
G25  = np.array([-31.77, -32.27, -33.51, -35.77, -38.67])     # kJ/mol
G37  = np.array([-32.16, -32.69, -34.00, -36.40, -39.49])     # kJ/mol
H_PRINTED = np.array([-22.08, -21.83, -46.17, -52.04, -18.30])

def gibbs_helmholtz(g1, g2, T1=T1, T2=T2):
    return (g2 / T2 - g1 / T1) / (1 / T2 - 1 / T1)

H = gibbs_helmholtz(G25, G37)
tab = pd.DataFrame({"pH": pH, "dG(25 C)": G25, "dG(37 C)": G37,
                    "dH recomputed": H, "dH printed": H_PRINTED,
                    "gap": H - H_PRINTED})
print(tab.to_string(index=False, formatters={
    "pH": "{:.1f}".format, "dG(25 C)": "{:.2f}".format, "dG(37 C)": "{:.2f}".format,
    "dH recomputed": "{:.2f}".format, "dH printed": "{:.2f}".format, "gap": "{:+.2f}".format}))

ok = np.abs(H - H_PRINTED) < 0.02
print(f"\nrows that reproduce: pH {[f'{v:g}' for v in pH[ok]]}")
print(f"rows that do not:    pH {[f'{v:g}' for v in pH[~ok]]}")
assert ok.sum() == 3 and (~ok).sum() == 2

 pH dG(25 C) dG(37 C) dH recomputed dH printed    gap
6.0   -31.77   -32.16        -22.08     -22.08  -0.00
6.5   -32.27   -32.69        -21.83     -21.83  -0.00
7.0   -33.51   -34.00        -21.34     -46.17 +24.83
7.5   -35.77   -36.40        -20.12     -52.04 +31.92
8.0   -38.67   -39.49        -18.30     -18.30  +0.00

rows that reproduce: pH ['6', '6.5', '8']
rows that do not:    pH ['7', '7.5']


### What the corrected column looks like, and the scatter it removes

The illustration closes: *"Note that there is considerable scatter in the results, which is
an indication of the difficulty in obtaining accurate thermodynamic data for some
biochemical reactions."*

**With the arithmetic corrected there is no scatter.** The heat of reaction runs
smoothly from $-22.1$ to $-18.3$ kJ/mol, monotone in pH, with monotonically growing steps.
The spread falls from 33.7 kJ/mol to 3.8. **The conclusion the illustration draws is an
artifact of two bad rows**, and it is the one sentence in it a reader would remember.

In [8]:
print("printed column:")
print(f"  values   {H_PRINTED}")
print(f"  spread   {H_PRINTED.max()-H_PRINTED.min():.2f} kJ/mol")
print(f"  monotone {bool(np.all(np.diff(H_PRINTED) > 0) or np.all(np.diff(H_PRINTED) < 0))}")
print("\nrecomputed column:")
print(f"  values   {np.round(H, 2)}")
print(f"  spread   {H.max()-H.min():.2f} kJ/mol")
print(f"  monotone {bool(np.all(np.diff(H) > 0))}")
print(f"  steps    {np.round(np.diff(H), 3)}  -- each larger than the last")

# The van 't Hoff picture: dH is the slope of dG/T against 1/T, and with only two
# temperatures it IS that slope, so a 1 kJ/mol error in either dG is worth about 25 kJ/mol
# in dH. That sensitivity is the real lesson available here, and it is quantitative.
sens = 1.0 / T2 / (1 / T2 - 1 / T1)
print(f"\nsensitivity: 1 kJ/mol of error in dG(37 C) moves dH by {abs(sens):.1f} kJ/mol")
print(f"             1 kJ/mol of error in dG(25 C) moves dH by "
      f"{abs(1.0/T1/(1/T2-1/T1)):.1f} kJ/mol")
print("Two temperatures 12 K apart amplify a 3 percent error in dG into a 100 percent")
print("error in dH. THAT is the difficulty with these data, and it needs no bad rows.")

printed column:
  values   [-22.08 -21.83 -46.17 -52.04 -18.3 ]
  spread   33.74 kJ/mol
  monotone False

recomputed column:
  values   [-22.08 -21.83 -21.34 -20.12 -18.3 ]
  spread   3.78 kJ/mol
  monotone True
  steps    [0.245 0.499 1.218 1.821]  -- each larger than the last

sensitivity: 1 kJ/mol of error in dG(37 C) moves dH by 24.8 kJ/mol
             1 kJ/mol of error in dG(25 C) moves dH by 25.8 kJ/mol
Two temperatures 12 K apart amplify a 3 percent error in dG into a 100 percent
error in dH. THAT is the difficulty with these data, and it needs no bad rows.


### Can the two printed values be reconstructed?

Tested rather than guessed, because a discrete reconstruction would say where the
slip happened and a failure to find one is itself information
[[enumerate-equation-variants]].

In [9]:
print("what input would give the printed dH, holding the other one fixed?")
for i in (2, 3):
    need_G37 = T2 * (H_PRINTED[i] * (1/T2 - 1/T1) + G25[i]/T1)
    need_G25 = T1 * (G37[i]/T2 - H_PRINTED[i] * (1/T2 - 1/T1))
    print(f"\n  pH {pH[i]}: printed dH = {H_PRINTED[i]:.2f}")
    print(f"    with dG(25) = {G25[i]:.2f} held, needs dG(37) = {need_G37:.3f}"
          f"   (table prints {G37[i]:.2f})")
    print(f"    with dG(37) = {G37[i]:.2f} held, needs dG(25) = {need_G25:.3f}"
          f"   (table prints {G25[i]:.2f})")

print("\npH 7.0 has a clean reconstruction:")
print(f"  dG(37) = -33.00 instead of -34.00 gives dH = "
      f"{gibbs_helmholtz(-33.51, -33.00):.2f}, the printed -46.17.")
print("  One digit, and -34 kJ/mol at pH 7 and 37 C is the value the section quotes in")
print("  its own text, so the TABLE is right and the dH was computed from a mistyped -33.")

print("\npH 7.5 does not:")
print(f"  it needs dG(37) = -35.115 or dG(25) = -37.005, and neither is a digit variant")
print(f"  of the printed -36.40 or -35.77. Nearby candidates:")
for g37 in (-35.40, -35.10, -35.12, -36.04):
    print(f"    dG(37) = {g37:7.2f} -> dH = {gibbs_helmholtz(-35.77, g37):8.2f}"
          f"   (printed -52.04)")
print("  NO discrete variant lands on -52.04. It is left unexplained rather than")
print("  fitted to, and the correct value is -20.12 either way.")

what input would give the printed dH, holding the other one fixed?

  pH 7.0: printed dH = -46.17
    with dG(25) = -33.51 held, needs dG(37) = -33.000   (table prints -34.00)
    with dG(37) = -34.00 held, needs dG(25) = -34.471   (table prints -33.51)

  pH 7.5: printed dH = -52.04
    with dG(25) = -35.77 held, needs dG(37) = -35.115   (table prints -36.40)
    with dG(37) = -36.40 held, needs dG(25) = -37.005   (table prints -35.77)

pH 7.0 has a clean reconstruction:
  dG(37) = -33.00 instead of -34.00 gives dH = -46.18, the printed -46.17.
  One digit, and -34 kJ/mol at pH 7 and 37 C is the value the section quotes in
  its own text, so the TABLE is right and the dH was computed from a mistyped -33.

pH 7.5 does not:
  it needs dG(37) = -35.115 or dG(25) = -37.005, and neither is a digit variant
  of the printed -36.40 or -35.77. Nearby candidates:
    dG(37) =  -35.40 -> dH =   -44.96   (printed -52.04)
    dG(37) =  -35.10 -> dH =   -52.42   (printed -52.04)
    dG(37) =  -35.1

## What the two illustrations say

Six statements, each asserted against the numbers.

In [10]:
checks = []
checks.append((f"13.3-2: the two routes to K0_AgCl agree -- {K_AGCL:.4e} here against"
               f" 1.6070e-10 from Illustration 13.2-3's five-point extrapolation",
               abs(K_AGCL - 1.6070e-10)/1.6070e-10 < 1e-3))
checks.append((f"13.3-2: the common ion cuts AgCl solubility by a factor of"
               f" {S_AGCL/C_Ag:.0f} -- three and a half orders of magnitude",
               S_AGCL/C_Ag > 1000))
checks.append((f"13.3-2: the illustration's shortcut is justified -- TlCl solubility moves"
               f" by {100*abs(C_Tl-S_TLCL)/S_TLCL:.1e} percent",
               abs(C_Tl - S_TLCL)/S_TLCL < 1e-6))
checks.append(("13.3-2: sqrt(0.144) = 0.3795, and 0.3975 reproduces all three printed"
               " numbers computed after it while 0.3795 reproduces the one before",
               abs(with_root(0.3975)["C_Ag"] - 2.119e-9)/2.119e-9 < 1e-3
               and abs(with_root(np.sqrt(0.144))["K_TlCl"] - 1.116e-2)/1.116e-2 < 1e-3))
checks.append((f"13.3-3: three of five printed heats reproduce exactly; two miss by"
               f" {np.abs((H-H_PRINTED)[~ok]).min():.1f} and"
               f" {np.abs((H-H_PRINTED)[~ok]).max():.1f} kJ/mol",
               ok.sum() == 3))
checks.append((f"13.3-3: corrected, the column is monotone and spans {H.max()-H.min():.1f}"
               f" kJ/mol, not {H_PRINTED.max()-H_PRINTED.min():.1f} -- the 'considerable"
               f" scatter' is the two bad rows",
               bool(np.all(np.diff(H) > 0))))
for text, okk in checks:
    print(f"  {'PASS' if okk else 'FAIL'}  {text}")
assert all(o for _, o in checks)

  PASS  13.3-2: the two routes to K0_AgCl agree -- 1.6070e-10 here against 1.6070e-10 from Illustration 13.2-3's five-point extrapolation
  PASS  13.3-2: the common ion cuts AgCl solubility by a factor of 6141 -- three and a half orders of magnitude
  PASS  13.3-2: the illustration's shortcut is justified -- TlCl solubility moves by 6.4e-07 percent
  PASS  13.3-2: sqrt(0.144) = 0.3795, and 0.3975 reproduces all three printed numbers computed after it while 0.3795 reproduces the one before
  PASS  13.3-3: three of five printed heats reproduce exactly; two miss by 24.8 and 31.9 kJ/mol
  PASS  13.3-3: corrected, the column is monotone and spans 3.8 kJ/mol, not 33.7 -- the 'considerable scatter' is the two bad rows


## Your turn

1. Illustration 13.3-2 assumes the thallium chloride solubility is unaffected and then
   checks it. **Construct a case where that shortcut fails.** Keep AgCl as one salt and
   replace TlCl with a chloride whose solubility is within a factor of ten of AgCl's; solve
   both ways and report where the shortcut first costs you a figure. What ratio of
   solubilities is the shortcut good for?
2. The ionic strength in Illustration 13.3-2 reaches 0.144 kmol/m$^3$, which is why
   Eq. 9.10-18 is used rather than the limiting law. Redo the whole illustration with the
   limiting law and report every number. How wrong is $K^{\circ}_{\rm TlCl}$, and how wrong
   is the final silver concentration? Which of the two is more sensitive, and why is it not
   the one with the larger ionic strength?
3. **Find the pH 7.5 slip in Illustration 13.3-3.** The notebook shows that pH 7.0's
   printed heat comes from a mistyped $-33.00$, but no digit variant reproduces pH 7.5's
   $-52.04$. Try: swapping the two temperatures; using $T$ in Celsius; using
   $\Delta H=\Delta G+T\Delta S$ with $\Delta S$ from the wrong row; and taking the pair
   from an adjacent row. Report what each gives. If none works, **say so** -- a wrong number
   does not owe you an explanation.
4. With only two temperatures, $\Delta_{\rm rxn}H^{\circ}$ is a difference of two nearly
   equal quantities. Propagate an uncertainty of $\pm0.1$ kJ/mol in each Gibbs energy
   through to the heat of reaction at every pH, and plot the result as error bars on the
   corrected column. Is the *corrected* trend -- $-22.1$ rising to $-18.3$ -- larger than
   its own uncertainty?
5. Eq. 13.3-10 says the equilibrium constant of a sum of reactions is the product of their
   constants. Verify it for a case you can check completely: take the water-gas shift as
   the difference of the two steam-carbon reactions in Sec. 13.3, compute $K_a$ for all
   three from Appendix A.IV at 1000 K, and confirm the relation to the digits the appendix
   supports.
6. The ATP reaction's Gibbs energy depends strongly on pH -- from $-31.8$ to $-38.7$ kJ/mol
   over two pH units. Where does that dependence come from, given that pH appears nowhere
   in the stoichiometry as written? Write the reaction with the proton explicit and use
   Sec. 13.5's ionization machinery to say how many protons are released, then check your
   answer against the slope of $\Delta_{\rm rxn}G^{\circ}$ against pH.